# Leniency sting experiment: replication notebook

Liar's Bar with one undercover agent and three ordinary agents. The undercover agent offers a "never challenge each other" pact, and four rules decide whether members can report it (`no_report`, `leniency_start`, `leniency_mid`, `reward_mid`).

This notebook:
1. installs the dependencies and runs the offline tests (no API cost),
2. rebuilds every table and plot from the shipped transcripts of the main study (no API cost),
3. optionally re-runs the full study (about $0.90 and about 70 minutes).

Re-runs will not match the shipped numbers exactly, because the model is not deterministic. Seeds, prompts and sampling are fixed.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/thubpham/collusion-hackathon"
RUN_FULL_STUDY = False

if not (Path.cwd() / "src" / "sting").exists():
    if not (Path.cwd() / "collusion-hackathon").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("collusion-hackathon")
print(Path.cwd())

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 1. Offline tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-400:])

## 2. Rebuild the tables and plots from the shipped transcripts

In [ ]:
RUN = "results/sting/main_llama70b"
subprocess.run([sys.executable, "analysis/sting_metrics.py", "--run", RUN], check=True)
subprocess.run([sys.executable, "analysis/sting_summary.py", "--run", RUN], check=True)

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = pd.read_csv(f"{RUN}/summary.csv")
counts = summary[summary["numerator"].notna()]
display(counts.pivot(index="metric", columns="arm", values="rate_text")[["no_report", "leniency_start", "leniency_mid", "reward_mid"]])
display(pd.read_csv(f"{RUN}/fisher_tests.csv").round(3))

In [ ]:
for name in ["exposure_by_arm", "report_rate_by_arm", "pact_length_by_arm", "report_timing_by_arm"]:
    display(Image(f"{RUN}/plots/{name}.png"))

## 3. Full re-run

Set `RUN_FULL_STUDY = True` in the first cell. Runs the four arms in parallel into `results/sting/replication`, then rebuilds the tables.

In [ ]:
if RUN_FULL_STUDY:
    if not os.environ.get("OPENROUTER_API_KEY"):
        from getpass import getpass
        os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")
    arms = ["no_report", "leniency_start", "leniency_mid", "reward_mid"]
    procs = [
        subprocess.Popen(
            [sys.executable, "scripts/run_sting.py", "--config", f"configs/sting/{a}__llama70b.json",
             "--games", "10", "--run-id", "replication"]
        )
        for a in arms
    ]
    for p in procs:
        p.wait()
    subprocess.run([sys.executable, "analysis/sting_metrics.py", "--run", "results/sting/replication"], check=True)
    subprocess.run([sys.executable, "analysis/sting_summary.py", "--run", "results/sting/replication"], check=True)